In [ ]:
import pandas as pd
df = pd.read_csv("placement_predict_50k Dataset.csv")

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

features = [c for c in df.columns if c not in ["PlacementStatus", "CGPA_Tier"]]
X = df[features]
y = df["PlacementStatus"]

num = X.select_dtypes(include=np.number).columns
cat = X.select_dtypes(exclude=np.number).columns

preprocessor = ColumnTransformer([
    ("num", Pipeline([
        ("imp", SimpleImputer(strategy="median")),
        ("sc", StandardScaler())
    ]), num),
    ("cat", Pipeline([
        ("imp", SimpleImputer(strategy="most_frequent")),
        ("enc", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
    ]), cat)
])

In [ ]:
def evaluate(model, X_train, X_val, y_train, y_val):
    model.fit(X_train, y_train)
    print("Train Accuracy:", model.score(X_train, y_train))
    print("Validation Accuracy:", model.score(X_val, y_val))
    print(classification_report(y_val, model.predict(X_val)))
    print(confusion_matrix(y_val, model.predict(X_val)))
    return model

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=.2, random_state=42, stratify=y
)

model1 = Pipeline([
    ("prep", preprocessor),
    ("lr", LogisticRegression(max_iter=5000))
])

model1 = evaluate(model1, X_train, X_val, y_train, y_val)

Train Accuracy: 0.997475
Validation Accuracy: 0.9968
              precision    recall  f1-score   support

           0       1.00      0.99      1.00      3429
           1       1.00      1.00      1.00      6571

    accuracy                           1.00     10000
   macro avg       1.00      1.00      1.00     10000
weighted avg       1.00      1.00      1.00     10000

[[3411   18]
 [  14 6557]]


In [ ]:
X2 = df[features]
y2 = df["CGPA_Tier"]

X2_train, X2_val, y2_train, y2_val = train_test_split(
    X2, y2, test_size=.2, random_state=42, stratify=y2
)

model2 = Pipeline([
    ("prep", preprocessor),
    ("lr", LogisticRegression(max_iter=5000))
])

model2 = evaluate(model2, X2_train, X2_val, y2_train, y2_val)

Train Accuracy: 0.983325
Validation Accuracy: 0.9855
              precision    recall  f1-score   support

        High       1.00      1.00      1.00      3336
         Low       0.98      0.98      0.98      3347
         Mid       0.98      0.97      0.98      3317

    accuracy                           0.99     10000
   macro avg       0.99      0.99      0.99     10000
weighted avg       0.99      0.99      0.99     10000

[[3327    0    9]
 [   0 3294   53]
 [   8   75 3234]]


In [ ]:
academic = [
    "CGPA", "AptitudeTestScore", "CodingTestScore",
    "SoftSkillsRating", "MockInterviewScore", "Workshops"
]

X3 = df[academic]
y3 = df["PlacementStatus"]

X3_train, X3_val, y3_train, y3_val = train_test_split(
    X3, y3, test_size=.2, random_state=42, stratify=y3
)

model3 = Pipeline([
    ("imp", SimpleImputer(strategy="median")),
    ("sc", StandardScaler()),
    ("lr", LogisticRegression(max_iter=5000))
])

model3 = evaluate(model3, X3_train, X3_val, y3_train, y3_val)

Train Accuracy: 0.889725
Validation Accuracy: 0.8838
              precision    recall  f1-score   support

           0       0.86      0.80      0.82      3429
           1       0.90      0.93      0.91      6571

    accuracy                           0.88     10000
   macro avg       0.88      0.86      0.87     10000
weighted avg       0.88      0.88      0.88     10000

[[2727  702]
 [ 460 6111]]


In [ ]:
comparison = pd.DataFrame({
    "Model": ["PlacementStatus", "CGPA_Tier", "Academic Only"],
    "Validation Accuracy": [
        model1.score(X_val, y_val),
        model2.score(X2_val, y2_val),
        model3.score(X3_val, y3_val)
    ]
})

comparison

,Model,Validation Accuracy
0,PlacementStatus,0.9967
1,CGPA_Tier,0.9855
2,Academic Only,0.8838
